In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA — 读取 & 预处理 & 写缓存（升级版）
#
# 新增：
#   * 在不改原 calc_parc_o3 的前提下，额外计算 1–100 hPa 的部分臭氧柱
#   * 所有输出（time series、lowest10、lowest25）各自独立缓存
#   * 用 PRESSURE_RANGES 自动循环，方便以后再加别的层
# ============================================================

import os, glob
import numpy as np
import xarray as xr

# ----------------- 已有的 calc_parc_o3 函数（保持不动） -----------------
def calc_parc_o3(var):
    m_air = 28.964/(6.022E23)
    g = 980.6

    if 'plev' in var.dims:
        plev = var.plev
        level = 'plev'
    if 'lev' in var.dims:
        plev = var.lev
        level = 'lev'
    if 'level' in var.dims:
        plev = var.level
        level = 'level'

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    if plev[0] < plev[len(plev)-1] and plev[len(plev)-1] <= 1000:
        factor = 100
        factor_2 = 1  # hPa → Pa
    if plev[0] > plev[len(plev)-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    if plev[0] < plev[len(plev)-1] and plev[len(plev)-1] > 1000:
        factor = 1
        factor_2 = 100
    if plev[0] > plev[len(plev)-1] and plev[0] > 1000:
        factor = 1
        factor_2 = 100

    # plev 升序
    if plev[0] < plev[len(plev)-1]:
        for levelx in range(1, len(plev)):
            delta_p[:, levelx].fill(plev[levelx] - plev[levelx-1])

        weights_p = xr.DataArray(delta_p*factor,
                                 dims=['time', level],
                                 coords=[time, plev])

        O3 = var * weights_p * 10 / (g * m_air)

        if level == 'plev':
            O3 = O3.sel(plev=slice(30*factor_2, 70*factor_2))
        if level == 'lev':
            O3 = O3.sel(lev=slice(30*factor_2, 70*factor_2))
        if level == 'level':
            O3 = O3.sel(level=slice(30*factor_2, 70*factor_2))

        O3 = O3.sum(dim=level)
        O3 = O3 / 2.687E16  # DU

    # plev 降序
    if plev[0] > plev[len(plev)-1]:
        for levelx in range(0, len(plev)-1):
            delta_p[:, levelx].fill(plev[levelx] - plev[levelx+1])

        weights_p = xr.DataArray(delta_p*factor,
                                 dims=['time', level],
                                 coords=[time, plev])

        O3 = var * weights_p * 10 / (g * m_air)

        if level == 'plev':
            O3 = O3.sel(plev=slice(70*factor_2, 30*factor_2))
        if level == 'lev':
            O3 = O3.sel(lev=slice(70*factor_2, 30*factor_2))
        if level == 'level':
            O3 = O3.sel(level=slice(70*factor_2, 30*factor_2))

        O3 = O3.sum(dim=level)
        O3 = O3 / 2.687E16

    return O3.where(O3 != 0)

# ----------------- 新增：可指定压强范围的通用版本 -----------------
def calc_parc_o3_range(var, p_top_hpa=30, p_bot_hpa=70):
    """
    计算 p_top_hpa – p_bot_hpa (hPa) 的部分臭氧柱（DU）
    逻辑与 calc_parc_o3 完全一致，只是切层范围可变。
    """
    m_air = 28.964/(6.022E23)
    g = 980.6

    if 'plev' in var.dims:
        plev = var.plev
        level = 'plev'
    elif 'lev' in var.dims:
        plev = var.lev
        level = 'lev'
    elif 'level' in var.dims:
        plev = var.level
        level = 'level'
    else:
        raise ValueError("Cannot find vertical dim (plev/lev/level).")

    time = var.time
    delta_p = np.zeros((len(time), len(plev)))

    if plev[0] < plev[-1] and plev[-1] <= 1000:
        factor = 100
        factor_2 = 1
    if plev[0] > plev[-1] and plev[0] <= 1000:
        factor = 100
        factor_2 = 1
    if plev[0] < plev[-1] and plev[-1] > 1000:
        factor = 1
        factor_2 = 100
    if plev[0] > plev[-1] and plev[0] > 1000:
        factor = 1
        factor_2 = 100

    # 升序
    if plev[0] < plev[-1]:
        for k in range(1, len(plev)):
            delta_p[:, k].fill(plev[k] - plev[k-1])

        weights_p = xr.DataArray(delta_p*factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)

        sl = slice(p_top_hpa*factor_2, p_bot_hpa*factor_2)
        O3 = O3.sel({level: sl})
        O3 = O3.sum(dim=level) / 2.687E16

    # 降序
    else:
        for k in range(0, len(plev)-1):
            delta_p[:, k].fill(plev[k] - plev[k+1])

        weights_p = xr.DataArray(delta_p*factor, dims=['time', level], coords=[time, plev])
        O3 = var * weights_p * 10 / (g * m_air)

        sl = slice(p_bot_hpa*factor_2, p_top_hpa*factor_2)
        O3 = O3.sel({level: sl})
        O3 = O3.sum(dim=level) / 2.687E16

    return O3.where(O3 != 0)

# ----------------- BlockA 主体 -----------------
ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
os.makedirs(ROOT_DIR, exist_ok=True)

# 极端年份缓存开关
USE_CACHED_EXTREMES = True

# 压强范围配置： (p_top, p_bot, tag)
PRESSURE_RANGES = [
    (30, 70, "30_70hPa"),
    (1, 100, "1_100hPa"),
]

# ---------- A1. 读取 200 年 EXTR O3 ----------
o3_pattern = "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/o3/CO2x1SmidEmin_yBWCN.cam.h1.????.O3.isobar.nc"
o3_files = sorted(glob.glob(o3_pattern))
print(f"[BlockA] Found {len(o3_files)} EXTR O3 files:")
if len(o3_files) > 0:
    print("  first:", o3_files[0])
    print("  last :", o3_files[-1])

ds_o3 = xr.open_mfdataset(o3_files, combine="by_coords", parallel=False)
O3_4d = ds_o3["O3"]

# ---------- A2. merged WACCM 读入 ----------
merged_path = "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc"
print(f"[BlockA] Reading merged WACCM file: {merged_path}")
ds_merged = xr.open_dataset(merged_path)
O3_merged_4d  = ds_merged["O3"]

for ptop, pbot, tag in PRESSURE_RANGES:

    print(f"\n[BlockA] ==== Processing range {ptop}-{pbot} hPa ({tag}) ====")

    # 输出文件路径（每个范围一套）
    PC_TS_EXTR_NC   = os.path.join(ROOT_DIR, f"O3_pc_EXTR_60_90N_{tag}_200yrs.nc")
    PC_TS_MERGED_NC = os.path.join(ROOT_DIR, f"O3_pc_MERGED_60_90N_{tag}.nc")
    LOW10_TXT       = os.path.join(ROOT_DIR, f"O3_lowest10_years_{tag}.txt")
    LOW25_TXT       = os.path.join(ROOT_DIR, f"O3_lowest25pct_years_{tag}.txt")

    # ===== EXTR：计算 partial column =====
    if ptop == 30 and pbot == 70:
        O3_parc = calc_parc_o3(O3_4d)
    else:
        O3_parc = calc_parc_o3_range(O3_4d, ptop, pbot)

    O3_zm   = O3_parc.mean(dim="lon")
    O3_cap  = O3_zm.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(O3_cap.lat))
    var_extr = O3_cap.weighted(weights).mean(dim="lat").load()
    var_extr.name = f"O3_pc_60_90N_{tag}"

    var_extr.to_netcdf(PC_TS_EXTR_NC)
    print(f"[BlockA] Saved EXTR polar-cap partial column TS to: {PC_TS_EXTR_NC}")

    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    print("[BlockA] EXTR years:", years_extr)

    # ===== extreme years（基于 3–4 月日值）=====
    if USE_CACHED_EXTREMES and os.path.exists(LOW10_TXT) and os.path.exists(LOW25_TXT):
        print("[BlockA] Using cached extreme-year txt files.")
    else:
        print("[BlockA] Computing extreme years from 3–4 month daily values ...")

        spring = var_extr.sel(time=var_extr.time.dt.month.isin([3, 4]))
        spring_min_by_year = spring.groupby("time.year").min("time")

        spring_years = spring_min_by_year["year"].values.astype(int)
        spring_min_vals = spring_min_by_year.values

        idx_sorted = np.argsort(spring_min_vals)

        n_low10 = min(10, len(spring_years))
        lowest10_years = spring_years[idx_sorted[:n_low10]]

        n_low25 = max(int(0.25 * len(spring_years)), 1)
        lowest25_years = spring_years[idx_sorted[:n_low25]]

        np.savetxt(LOW10_TXT, lowest10_years, fmt="%04d")
        np.savetxt(LOW25_TXT, lowest25_years, fmt="%04d")

        print(f"[BlockA] Saved lowest-10 years to: {LOW10_TXT}")
        print(f"[BlockA] Saved lowest-25% years to: {LOW25_TXT}")

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)

    print("[BlockA] Lowest 10 years (EXTR):", lowest10_years)
    print("[BlockA] Lowest 25% years (EXTR):", lowest25_years)

    # ===== merged：计算 partial column =====
    if ptop == 30 and pbot == 70:
        O3_merged_parc = calc_parc_o3(O3_merged_4d)
    else:
        O3_merged_parc = calc_parc_o3_range(O3_merged_4d, ptop, pbot)

    O3_merged_zm   = O3_merged_parc.mean(dim="lon")
    O3_merged_cap  = O3_merged_zm.sel(lat=slice(60, 90))
    weights_m      = np.cos(np.deg2rad(O3_merged_cap.lat))

    var_merged = O3_merged_cap.weighted(weights_m).mean(dim="lat").load()
    var_merged.name = f"O3_pc_60_90N_{tag}"

    var_merged.to_netcdf(PC_TS_MERGED_NC)
    print(f"[BlockA] Saved merged polar-cap partial column TS to: {PC_TS_MERGED_NC}")

    years_merged = np.unique(var_merged.time.dt.year.values.astype(int))
    print("[BlockA] merged years:", years_merged)

print("\n[BlockA] Done.")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB — 绘图（升级版）
#   对 PRESSURE_RANGES 中每个范围
#     * 图1：lowest10 vs 其他年气候态±std
#     * 图2：merged special years vs EXTR climatology & low25 composite
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
os.makedirs(ROOT_DIR, exist_ok=True)

N_PREV = 91  # Oct–Dec

PRESSURE_RANGES = [
    (30, 70, "30_70hPa"),
    (1, 100, "1_100hPa"),
]

for ptop, pbot, tag in PRESSURE_RANGES:

    print(f"\n[BlockB] ==== Plotting range {ptop}-{pbot} hPa ({tag}) ====")

    PC_TS_EXTR_NC   = os.path.join(ROOT_DIR, f"O3_pc_EXTR_60_90N_{tag}_200yrs.nc")
    PC_TS_MERGED_NC = os.path.join(ROOT_DIR, f"O3_pc_MERGED_60_90N_{tag}.nc")
    LOW10_TXT       = os.path.join(ROOT_DIR, f"O3_lowest10_years_{tag}.txt")
    LOW25_TXT       = os.path.join(ROOT_DIR, f"O3_lowest25pct_years_{tag}.txt")

    # ============================================================
    # B1. 读取 EXTR time series & 极端年列表
    # ============================================================
    var_extr = xr.load_dataarray(PC_TS_EXTR_NC)
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)

    print("[BlockB] EXTR years:", years_extr)
    print("[BlockB] Lowest 10 years (EXTR):", lowest10_years)
    print("[BlockB] Lowest 25% years (EXTR):", lowest25_years)

    n_days = int(var_extr.time.dt.dayofyear.max())
    print("[BlockB] Days per year assumed:", n_days)

    # ============================================================
    # B2. 图1：10 个极端低年 + 其他年气候态包络
    # ============================================================
    fig1, ax1 = plt.subplots(1, 1, figsize=(13, 10), constrained_layout=True)

    low_years = lowest10_years
    neutral_years = years_extr[~np.isin(years_extr, low_years)]

    var_low_cur   = var_extr.sel(time=var_extr.time.dt.year.isin(low_years))
    var_low_prev  = var_extr.sel(time=var_extr.time.dt.year.isin(low_years - 1))

    var_neu_cur   = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years))
    var_neu_prev  = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years - 1))

    neu_cur_mean = np.array(var_neu_cur.groupby("time.dayofyear").mean("time"))
    neu_cur_std  = np.array(var_neu_cur.groupby("time.dayofyear").std("time"))
    neu_prev_mean = np.array(var_neu_prev.groupby("time.dayofyear").mean("time"))
    neu_prev_std  = np.array(var_neu_prev.groupby("time.dayofyear").std("time"))

    n_extreme = len(low_years)
    var_low_cur_arr  = np.reshape(np.array(var_low_cur), (n_extreme, n_days))
    var_low_prev_arr = np.reshape(np.array(var_low_prev), (n_extreme, n_days))

    colors_ext = cm.twilight(np.linspace(0, 1, n_extreme))

    for j in range(n_extreme):
        ax1.plot(
            range(N_PREV, n_days),
            var_low_cur_arr[j, 0:n_days-N_PREV],
            color=colors_ext[j], alpha=0.8, linewidth=2,
            label="low O3 years" if j == 0 else None,
        )
        ax1.plot(
            range(0, N_PREV),
            var_low_prev_arr[j, n_days-N_PREV:n_days],
            color=colors_ext[j], alpha=0.8, linewidth=2,
        )

    ax1.errorbar(
        range(N_PREV, n_days),
        neu_cur_mean[0:n_days-N_PREV],
        neu_cur_std[0:n_days-N_PREV],
        color="grey", alpha=0.5, elinewidth=1.5, linewidth=3,
        label="all other years",
    )
    ax1.errorbar(
        range(0, N_PREV),
        neu_prev_mean[n_days-N_PREV:n_days],
        neu_prev_std[n_days-N_PREV:n_days],
        color="grey", alpha=0.5, elinewidth=1.5, linewidth=3,
    )

    ax1.set_xlim(0, n_days)
    ax1.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax1.set_xticklabels(
        ["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","June","July","Aug","Sep"],
        fontsize=15
    )
    ax1.set_ylabel(f"Partial ozone column, {ptop}–{pbot} hPa (DU)", fontsize=18)
    ax1.set_yticklabels(ax1.get_yticks(), size=18)
    ax1.legend(fontsize=14)

    out_png1 = os.path.join(ROOT_DIR, f"O3_evolution_min_10extr_200yrs_{tag}.png")
    out_pdf1 = os.path.join(ROOT_DIR, f"O3_evolution_min_10extr_200yrs_{tag}.pdf")
    plt.savefig(out_png1, dpi=300)
    plt.savefig(out_pdf1)
    plt.show()

    print("[BlockB] Saved fig1 to:")
    print("   ", out_png1)
    print("   ", out_pdf1)

    # ============================================================
    # B3. 图2：special years vs climatology & low25 composite
    # ============================================================
    var_merged = xr.load_dataarray(PC_TS_MERGED_NC)
    years_merged = np.unique(var_merged.time.dt.year.values.astype(int))
    years_special = np.array([8, 13, 14, 19], dtype=int)

    clim_all_day_mean = var_extr.groupby("time.dayofyear").mean("time")
    clim_all_day_std  = var_extr.groupby("time.dayofyear").std("time")

    base_low25_cur  = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years))
    base_low25_prev = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years - 1))

    clim_low25_cur_mean  = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_cur_std   = base_low25_cur.groupby("time.dayofyear").std("time")
    clim_low25_prev_mean = base_low25_prev.groupby("time.dayofyear").mean("time")
    clim_low25_prev_std  = base_low25_prev.groupby("time.dayofyear").std("time")

    clim_low25_day_mean = clim_low25_cur_mean.copy()
    clim_low25_day_std  = clim_low25_cur_std.copy()

    oct_dec = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)
    clim_low25_day_mean.loc[dict(dayofyear=oct_dec)] = clim_low25_prev_mean.sel(dayofyear=oct_dec).values
    clim_low25_day_std.loc[dict(dayofyear=oct_dec)]  = clim_low25_prev_std.sel(dayofyear=oct_dec).values

    clim_all_day_mean = np.array(clim_all_day_mean)
    clim_all_day_std  = np.array(clim_all_day_std)
    clim_low25_day_mean = np.array(clim_low25_day_mean)
    clim_low25_day_std  = np.array(clim_low25_day_std)

    all_comp_mean = np.concatenate([clim_all_day_mean[n_days-N_PREV:n_days],
                                    clim_all_day_mean[0:n_days-N_PREV]])
    all_comp_std  = np.concatenate([clim_all_day_std[n_days-N_PREV:n_days],
                                    clim_all_day_std[0:n_days-N_PREV]])

    low25_comp_mean = np.concatenate([clim_low25_day_mean[n_days-N_PREV:n_days],
                                      clim_low25_day_mean[0:n_days-N_PREV]])
    low25_comp_std  = np.concatenate([clim_low25_day_std[n_days-N_PREV:n_days],
                                      clim_low25_day_std[0:n_days-N_PREV]])

    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig2, ax2 = plt.subplots(1, 1, figsize=(6.5, 4.0), constrained_layout=True)
    x_full = np.arange(n_days)

    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(years_special)))

    for i, y in enumerate(years_special):
        if y not in years_merged:
            continue

        cur  = var_merged.sel(time=var_merged.time.dt.year == y)
        prev = var_merged.sel(time=var_merged.time.dt.year == (y - 1))
        if (cur.size < n_days) or (prev.size < n_days):
            continue

        series_comp = np.concatenate([
            np.array(prev)[n_days-N_PREV:n_days],
            np.array(cur)[0:n_days-N_PREV],
        ])

        ax2.plot(x_full, series_comp, lw=1.5, color=colors_spec[i], label=f"Year {y:04d}")

    ax2.fill_between(x_full, all_comp_mean-all_comp_std, all_comp_mean+all_comp_std,
                     color="0.85", alpha=0.9, linewidth=0, zorder=0)
    ax2.plot(x_full, all_comp_mean, ls="--", lw=1.8, color="black",
             label="EXTR climatology (all years)")

    ax2.fill_between(x_full, low25_comp_mean-low25_comp_std, low25_comp_mean+low25_comp_std,
                     facecolor="none", edgecolor="0.5", hatch="///",
                     linewidth=0.0, zorder=1)
    ax2.plot(x_full, low25_comp_mean, ls="-", lw=2.0, color="black",
             label="EXTR low-ozone composite (bottom 25%)")

    ax2.set_xlim(0, n_days)
    ax2.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax2.set_xticklabels(
        ["Oct","Nov","Dec","Jan","Feb","Mar","Apr","May","June","July","Aug","Sep"]
    )
    ax2.set_ylabel(f"Partial O$_3$ column, {ptop}–{pbot} hPa (DU)")
    ax2.set_title(f"Polar cap (60–90°N) partial ozone column ({ptop}–{pbot} hPa)")
    ax2.grid(False)

    patch_all = Patch(facecolor="0.85", edgecolor="none", label="EXTR all-year ±1σ")
    patch_low = Patch(facecolor="none", edgecolor="0.5", hatch="///", label="EXTR low-ozone ±1σ")
    line_all  = Line2D([0],[0], color="black", lw=1.8, ls="--", label="EXTR climatology")
    line_low  = Line2D([0],[0], color="black", lw=2.0, ls="-",  label="EXTR low-ozone composite")

    handles_years = [Line2D([0],[0], color=colors_spec[i], lw=1.5, label=f"Year {y:04d}")
                     for i, y in enumerate(years_special)]
    handles = handles_years + [line_all, patch_all, line_low, patch_low]
    ax2.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    out_png2 = os.path.join(ROOT_DIR, f"O3_daily_special_years_vs_extr_climatology_{tag}.png")
    out_pdf2 = os.path.join(ROOT_DIR, f"O3_daily_special_years_vs_extr_climatology_{tag}.pdf")
    plt.savefig(out_png2, dpi=300)
    plt.savefig(out_pdf2)
    plt.show()

    print("[BlockB] Saved fig2 to:")
    print("   ", out_png2)
    print("   ", out_pdf2)

print("\n[BlockB] Done.")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# ============================================================
# BlockC — 计算 O3 垂直时间剖面 anomaly + 显著性（t 检验 + bootstrap）
#
# 输出一个 NetCDF：O3_vert_anom_special.nc
# 包含：
#  - coords:
#      ref_year (n_ref) : [8, 13, 14, 19]
#      lev : 垂直层（来自 merged 文件，统一命名为 lev）
#      time_index : 0..364 （拼接后的时间）
#      dayofyear : 对应的 DOY 序列，如 [275..365, 1..274]
#  - variables (ppm):
#      anom_all (ref_year, lev, time_index)
#      anom_low25 (ref_year, lev, time_index)
#      clim_all_comp (lev, time_index)
#      clim_low25_comp (lev, time_index)
#  - 显著性掩膜（True = 显著）：
#      t_sig_all (ref_year, lev, time_index)
#      b_sig_all (ref_year, lev, time_index)
#      t_sig_low25 (ref_year, lev, time_index)
#      b_sig_low25 (ref_year, lev, time_index)
#
# 注：本版本修正了 EXTREME-25% 气候态的构造方式，确保：
#   - Oct–Dec 使用 “EXTREME25% 这些年前一年的 Oct–Dec” 的平均
#   - Jan–Sep 使用 “EXTREME25% 这些年的当年 Jan–Sep” 的平均
#   与 BlockB 中的时间拼接一致，从而避免 1.1 与 12.31 的人为跳跃。
# ============================================================

import os
import glob
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
os.makedirs(ROOT_DIR, exist_ok=True)

# EXTR 200 年 O3 文件模式
EXTR_O3_PATTERN = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/o3/"
    "CO2x1SmidEmin_yBWCN.cam.h1.????.O3.isobar.nc"
)

# merged WACCM 文件
MERGED_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)

# 极端年 txt（来自 BlockA）
LOW25_TXT = os.path.join(ROOT_DIR, "O3_lowest25pct_years.txt")

# 输出 NC
OUT_NC = os.path.join(ROOT_DIR, "O3_vert_anom_special.nc")

# 拼接窗口参数：前一年拼 91 天（Oct–Dec）
N_PREV = 91

# 特殊年份（merged 中的 0–100 年）
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


# ----------------- 计算极帽平均垂直 O3 的工具函数 -----------------
def calc_polar_cap_o3_lev(ds, var="O3"):
    """
    从 ds 中提取 O3，做：
      - 经向平均
      - 60–90°N 余弦加权
    返回 (time, lev) 的 DataArray，单位保持为 mol/mol。
    """
    da = ds[var]

    # 先经向平均
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    # 纬向极帽平均（60–90N）
    da = da.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(da.lat))
    da = da.weighted(weights).mean(dim="lat")

    # 确保有垂直维
    if "plev" in da.dims:
        lev_name = "plev"
    elif "lev" in da.dims:
        lev_name = "lev"
    elif "level" in da.dims:
        lev_name = "level"
    else:
        raise ValueError(
            f"O3 has no vertical dim among ('plev','lev','level'), dims={da.dims}"
        )

    # 垂直维统一命名为 lev
    if lev_name != "lev":
        da = da.rename({lev_name: "lev"})

    return da  # (time, lev)


def bootstrap_ci(data, nboot=5000, alpha=0.05, rng=None):
    """
    data: 1D array-like 样本（可以是基准 anomalies）
    返回：bootstrap 的均值置信区间 [low, high]
    """
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)

    low = np.percentile(means, 100 * alpha / 2.0)
    high = np.percentile(means, 100 * (1 - alpha / 2.0))
    return low, high


def compute_significance_for_baseline(
    base_anom,   # (time, lev) DataArray, EXTR anomalies
    anom_ref,    # (lev, time_index) np.array, ref year anomaly（mol/mol）
    doy_base,    # (time,) DOY of base
    doy_comp,    # (time_index,) DOY for composite timeline
    alpha=0.05,
    nboot=5000,
):
    """
    对给定基准样本 base_anom 和某一特定年份的 anomaly(anom_ref)，
    在每个 (lev, time_index) 上做：
      - t 检验：obs vs base_anom[同一 DOY 的样本分布]，检验均值是否为 0
      - bootstrap CI：同一日的样本 anomalies 上 bootstrap 均值 CI，obs 是否落在 CI 外

    返回：
      t_sig (lev, time_index) — True = 显著
      b_sig (lev, time_index) — True = 显著
    """
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        # 找出基准样本中该 DOY 对应的所有时间索引
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue

        # shape: (n_samples_for_day, lev)
        day_samples = base_vals[mask_day, :]

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]

            # --- t 检验：假设样本分布均值为 0 ---
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            # --- bootstrap CI：基于 col 的均值 CI，看 obs 是否落在 CI 外 ---
            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig


# ----------------- BlockC 主体 -----------------
def main_blockC():

    # ===== 0. 读取 low-25% 年列表 =====
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockC] Lowest 25% EXTR years:", lowest25_years)

    # ===== 1. 读取 EXTR O3，计算垂直极帽平均 =====
    extr_files = sorted(glob.glob(EXTR_O3_PATTERN))
    print(f"[BlockC] Found {len(extr_files)} EXTR O3 files.")
    if len(extr_files) == 0:
        raise RuntimeError("No EXTR O3 files found, pattern: " + EXTR_O3_PATTERN)

    ds_extr = xr.open_mfdataset(extr_files, combine="by_coords", parallel=False)
    o3_extr_lev = calc_polar_cap_o3_lev(ds_extr, "O3")  # (time, lev), mol/mol
    ds_extr.close()

    # 时间信息
    doy_extr = o3_extr_lev.time.dt.dayofyear.values.astype(int)
    years_extr = o3_extr_lev.time.dt.year.values.astype(int)
    print("[BlockC] EXTR years:", np.unique(years_extr))

    # 假定 noleap：得到一年天数
    n_days = int(o3_extr_lev.time.dt.dayofyear.max())
    print("[BlockC] Days per year (EXTR):", n_days)

    # ===== 2. 计算 EXTR 全部年份 / 低25% 年日气候态 (mol/mol) =====

    # ---- 全部年份气候态（不需要修改） ----
    clim_all = o3_extr_lev.groupby("time.dayofyear").mean("time")  # (dayofyear, lev)

    # ---- [FIX] 低 25% 年气候态：前一年 Oct–Dec + 当年 Jan–Sep ----
    #
    # 思路：
    #   - base_low25_cur: 极端年的 "当年"（用于 Jan–Sep）
    #   - base_low25_prev: 极端年的 "前一年"（用于 Oct–Dec）
    #   - 先分别对 dayofyear 做 climatology，再在 dayofyear 维度上：
    #       * DOY 1..(n_days-N_PREV) 使用当前年的 climatology
    #       * DOY (n_days-N_PREV+1)..n_days 使用前一年的 climatology
    #   这样得到一个完整的 daily climatology（clim_low25_full），
    #   与后面 composite 的拼接逻辑完全一致。
    # --------------------------------------------------------------
    base_low25_cur = o3_extr_lev.sel(time=o3_extr_lev.time.dt.year.isin(lowest25_years))
    base_low25_prev = o3_extr_lev.sel(
        time=o3_extr_lev.time.dt.year.isin(lowest25_years - 1)
    )

    clim_low25_cur = base_low25_cur.groupby("time.dayofyear").mean("time")   # (dayofyear, lev)
    clim_low25_prev = base_low25_prev.groupby("time.dayofyear").mean("time") # (dayofyear, lev)

    # 先复制一份当前年的 climatology
    clim_low25_full = clim_low25_cur.copy()

    # Oct–Dec 对应的 dayofyear 范围（例如 noleap 时 275..365）
    oct_dec = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)

    # 用前一年的 climatology 替换 Oct–Dec 段
    clim_low25_full.loc[dict(dayofyear=oct_dec)] = (
        clim_low25_prev.sel(dayofyear=oct_dec).values
    )

    # 对应 DOY 序列（1..365）
    doys = clim_all["dayofyear"].values.astype(int)

    # ===== 3. 读取 merged O3 垂直极帽平均 =====
    print(f"[BlockC] Reading merged file: {MERGED_FILE}")
    ds_merged = xr.open_dataset(MERGED_FILE)
    o3_merged_lev = calc_polar_cap_o3_lev(ds_merged, "O3")  # (time, lev), mol/mol
    ds_merged.close()

    years_merged = o3_merged_lev.time.dt.year.values.astype(int)
    print("[BlockC] merged years:", np.unique(years_merged))

    # 垂直坐标（统一为 lev）
    lev = o3_merged_lev["lev"].values
    lev_n = lev.size

    # ===== 4. 构造拼接用 dayofyear 序列：前一年末 N_PREV + 当年开头 =====
    # 例如 365 天时： [275..365] + [1..274]
    doy_comp = np.concatenate(
        [
            np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),
            np.arange(1, n_days - N_PREV + 1, dtype=int),
        ]
    )
    t_len = doy_comp.size  # 应该是 n_days
    print("[BlockC] Composite DOY sequence:", doy_comp[:5], "...", doy_comp[-5:])

    # ===== 5. 计算 EXTR anomalies（相对于各自基准） =====
    # 这里 base_anom_all / base_anom_low25 用于显著性检验

    # 全部年份基准：clim_all
    clim_all_sel_for_extr = clim_all.sel(dayofyear=o3_extr_lev.time.dt.dayofyear)
    base_anom_all = o3_extr_lev - clim_all_sel_for_extr  # (time, lev)

    # [FIX] 低25% 年基准：使用 clim_low25_full（已包含 “前一年 Oct–Dec + 当年 Jan–Sep”）
    clim_low25_sel_for_extr = clim_low25_full.sel(
        dayofyear=o3_extr_lev.time.dt.dayofyear
    )
    base_anom_low25 = o3_extr_lev - clim_low25_sel_for_extr  # (time, lev)

    # ===== 6. 计算拼接后的基准气候态剖面（与 doy_comp 对齐） =====
    # 这些不依赖具体年份，只依赖 DOY_comp

    clim_all_comp = clim_all.sel(dayofyear=doy_comp)          # (time_index, lev)
    # [FIX] low25 使用 clim_low25_full，而非简单的 “极端年当年的 climatology”
    clim_low25_comp = clim_low25_full.sel(dayofyear=doy_comp) # (time_index, lev)

    # 转成 (lev, time_index) 方便后面和 anomaly 一起操作
    clim_all_comp_vals = clim_all_comp.transpose("lev", "dayofyear").values   # (lev, t_len)
    clim_low25_comp_vals = clim_low25_comp.transpose("lev", "dayofyear").values  # (lev, t_len)

    # ===== 7. 为每个 special year 计算 anomaly + 显著性 =====
    n_ref = len(YEARS_SPECIAL)

    # 存储数组（统一用 ppm）
    anom_all_ppm = np.zeros((n_ref, lev_n, t_len))
    anom_low25_ppm = np.zeros((n_ref, lev_n, t_len))

    t_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    t_sig_low25 = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_low25 = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        if y not in years_merged:
            print(f"[BlockC] ⚠️ Year {y:04d} not found in merged, skip.")
            continue

        # 当前年 & 前一年
        ref_cur = o3_merged_lev.sel(time=o3_merged_lev.time.dt.year == y)
        ref_prev = o3_merged_lev.sel(time=o3_merged_lev.time.dt.year == (y - 1))

        if ref_cur.time.size < n_days or ref_prev.time.size < n_days:
            print(
                f"[BlockC] ⚠️ Year {y:04d} or {y-1:04d} does not have {n_days} days, skip."
            )
            continue

        ref_cur_vals = np.array(ref_cur.transpose("time", "lev").values)   # (365, lev_n)
        ref_prev_vals = np.array(ref_prev.transpose("time", "lev").values) # (365, lev_n)

        # 前一年最后 N_PREV 天 + 当年最前 n_days-N_PREV 天
        ref_comp_vals = np.concatenate(
            [
                ref_prev_vals[n_days - N_PREV : n_days, :],    # (N_PREV, lev_n)
                ref_cur_vals[0 : n_days - N_PREV, :],          # (n_days-N_PREV, lev_n)
            ],
            axis=0,
        )  # (t_len, lev_n)

        # 转成 (lev, time_index)
        ref_comp = ref_comp_vals.T  # (lev_n, t_len)

        # --- anomaly（mol/mol） ---
        anom_all = ref_comp - clim_all_comp_vals       # (lev_n, t_len)
        anom_low = ref_comp - clim_low25_comp_vals     # (lev_n, t_len)

        # --- 转为 ppm 存储 ---
        anom_all_ppm[yi, :, :] = anom_all * 1e6
        anom_low25_ppm[yi, :, :] = anom_low * 1e6

        # 显著性检验：基于 EXTR anomalies（mol/mol）
        print(f"[BlockC] Computing significance for year {y:04d} vs ALL baseline ...")
        t_all, b_all = compute_significance_for_baseline(
            base_anom_all,   # EXTR anomalies vs all-years climatology
            anom_all,        # 该 ref 年 vs all-years 的 anomaly（mol/mol）
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )

        print(f"[BlockC] Computing significance for year {y:04d} vs LOW25 baseline ...")
        t_low, b_low = compute_significance_for_baseline(
            base_anom_low25, # EXTR anomalies vs low25 climatology
            anom_low,        # 该 ref 年 vs low25 的 anomaly（mol/mol）
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )

        t_sig_all[yi, :, :] = t_all
        b_sig_all[yi, :, :] = b_all
        t_sig_low25[yi, :, :] = t_low
        b_sig_low25[yi, :, :] = b_low

    # ===== 8. 保存到 NetCDF =====
    # 把 climatology 也转成 ppm
    clim_all_comp_ppm = clim_all_comp_vals * 1e6
    clim_low25_comp_ppm = clim_low25_comp_vals * 1e6

    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", YEARS_SPECIAL),
            "lev": ("lev", lev),
            "time_index": ("time_index", np.arange(t_len)),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            "anom_all": (("ref_year", "lev", "time_index"), anom_all_ppm),
            "anom_low25": (("ref_year", "lev", "time_index"), anom_low25_ppm),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_ppm),
            "clim_low25_comp": (("lev", "time_index"), clim_low25_comp_ppm),
            "t_sig_all": (("ref_year", "lev", "time_index"), t_sig_all),
            "b_sig_all": (("ref_year", "lev", "time_index"), b_sig_all),
            "t_sig_low25": (("ref_year", "lev", "time_index"), t_sig_low25),
            "b_sig_low25": (("ref_year", "lev", "time_index"), b_sig_low25),
        },
    )

    ds_out["anom_all"].attrs["units"] = "ppm"
    ds_out["anom_low25"].attrs["units"] = "ppm"
    ds_out["clim_all_comp"].attrs["units"] = "ppm"
    ds_out["clim_low25_comp"].attrs["units"] = "ppm"

    ds_out.attrs["description"] = (
        "O3 polar-cap (60–90N) time–height anomalies in ppm for special years "
        "relative to EXTR all-years climatology and low-25% climatology. "
        "Low-25% climatology is constructed using previous-year Oct–Dec and "
        "current-year Jan–Sep for the lowest-25% years. "
        "Significance masks (t-test & bootstrap) included."
    )

    ds_out.to_netcdf(OUT_NC)
    print(f"[BlockC] Saved vertical anomaly dataset to: {OUT_NC}")
    print("[BlockC] Done.")


if __name__ == "__main__":
    main_blockC()



In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD — 绘图：
#   使用 BlockC 计算好的数据，绘制时间×垂直剖面 anomaly（ppm）：
#
#   对每个 ref_year（8, 13, 14, 19）：
#     1) 相对于 EXTR 全部年份气候态：
#         - t 检验 “不显著区域” hatch 图
#         - bootstrap “不显著区域” hatch 图
#     2) 相对于 EXTR 低 25% 气候态：
#         - t 检验 “不显著区域” hatch 图
#         - bootstrap “不显著区域” hatch 图
#
#   横坐标：0..364，与 BlockB 一致，刻度 Oct–Sep
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
os.makedirs(ROOT_DIR, exist_ok=True)

IN_NC = os.path.join(ROOT_DIR, "O3_vert_anom_special.nc")

# 时间轴设置（与 BlockB 保持一致）
XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
                "May", "June", "July", "Aug", "Sep"]


# === 全局样式 ===
mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rc("font", family="sans-serif")
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
mpl.rc("axes", titlesize=11, labelsize=9, linewidth=0.8)
mpl.rc("legend", fontsize=7, frameon=False)
mpl.rc("figure", figsize=(6, 4), dpi=300)
mpl.rc("savefig", bbox="tight", pad_inches=0.1)


def guess_p_2_pa(lev_vals):
    """
    根据 lev 数值猜测是否是 hPa 或 Pa：
      - 若 max(lev) <= 2000 → 当作 hPa，乘 100 得到 Pa
      - 否则直接当作 Pa
    """
    lev_vals = np.asarray(lev_vals)
    if np.nanmax(lev_vals) <= 2000:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"


def plot_time_height_anom(
    x_idx,              # 1D, time_index 0..364
    lev_vals,           # 1D, lev coordinate
    anom_vals,          # 2D, (lev, time) anomaly in ppm
    clim_vals,          # 2D, (lev, time) baseline climatology in ppm
    nonsig_mask,        # 2D, (lev, time), True = not significant
    ref_year,
    baseline_label,
    test_label,         # 't-test' or 'bootstrap'
    outfile,
):
    """
    绘制一张时间×高度 anomaly 图：
      - 填色为 anomaly（ppm）
      - 黑色等值线为 baseline climatology（ppm）
      - 用 /// hatch 标记 “不显著区域”（nonsig_mask=True）
    """

    fig, ax = plt.subplots()

    # x 轴：0..364
    x = np.asarray(x_idx)

    # y 轴：转换为 Pa 然后 log
    p_pa, p_unit = guess_p_2_pa(lev_vals)

    # 填色范围（这里简单用 ±1.5 ppm，你可以后续按数据调）
    vlim = np.nanmax(np.abs(anom_vals))
    if np.isfinite(vlim) and vlim > 0:
        vmax = min(vlim, 2.0)  # 不让色阶太夸张
    else:
        vmax = 1.5
    levels = np.linspace(-vmax, vmax, 31)

    cf = ax.contourf(
        x, p_pa, anom_vals,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        antialiased=True,
        alpha=0.85,
    )

    # 叠加 baseline climatology 等值线
    # 可以根据典型值选一组等值线，或者自动生成
    clim_mean = np.nanmean(clim_vals)
    clim_std  = np.nanstd(clim_vals)
    if np.isfinite(clim_mean) and np.isfinite(clim_std) and clim_std > 0:
        clim_levels = np.linspace(clim_mean - 1.5*clim_std,
                                  clim_mean + 1.5*clim_std, 7)
    else:
        clim_levels = 7

    CS = ax.contour(
        x, p_pa, clim_vals,
        levels=clim_levels,
        colors="k",
        linewidths=0.7,
    )
    ax.clabel(CS, inline=True, fontsize=6, fmt="%.2f")

    # 不显著区域 hatch 标记
    if nonsig_mask is not None:
        mask_int = nonsig_mask.astype(int)
        ax.contourf(
            x, p_pa, mask_int,
            levels=[0.5, 1.5],
            colors="none",
            hatches=["///"],
            alpha=0,
        )
        patch = Patch(
            facecolor="white",
            hatch="///",
            edgecolor="black",
            label="Not significant (p ≥ 0.05)",
        )
        ax.legend(handles=[patch], loc="upper right")

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel(f"Pressure ({p_unit})")

    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)

    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.5)

    ax.set_title(
        f"O$_3$ anomaly (ppm), Year {int(ref_year):04d}\n"
        f"Baseline: {baseline_label}, Mask: non-significant by {test_label}"
    )

    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("O$_3$ anomaly (ppm)")

    plt.savefig(outfile, dpi=300)
    plt.close(fig)
    print("  Saved:", outfile)


def main_blockD():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_index = ds["time_index"].values

    anom_all = ds["anom_all"].values        # (ref_year, lev, time)
    anom_low = ds["anom_low25"].values     # (ref_year, lev, time)
    clim_all = ds["clim_all_comp"].values  # (lev, time)
    clim_low = ds["clim_low25_comp"].values

    t_sig_all = ds["t_sig_all"].values
    b_sig_all = ds["b_sig_all"].values
    t_sig_low = ds["t_sig_low25"].values
    b_sig_low = ds["b_sig_low25"].values

    ds.close()

    for yi, y in enumerate(ref_years):
        # 1) baseline = all-years climatology
        #   t 检验不显著区域
        nonsig_t_all = ~t_sig_all[yi, :, :]   # (lev, time)
        outfile = os.path.join(
            ROOT_DIR,
            f"O3_anom_year{int(y):04d}_vsALL_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_t_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="t-test",
            outfile=outfile,
        )

        #   bootstrap 不显著区域
        nonsig_b_all = ~b_sig_all[yi, :, :]
        outfile = os.path.join(
            ROOT_DIR,
            f"O3_anom_year{int(y):04d}_vsALL_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_b_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

        # 2) baseline = low-25% climatology
        #   t 检验不显著区域
        nonsig_t_low = ~t_sig_low[yi, :, :]
        outfile = os.path.join(
            ROOT_DIR,
            f"O3_anom_year{int(y):04d}_vsLOW25_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_t_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="t-test",
            outfile=outfile,
        )

        #   bootstrap 不显著区域
        nonsig_b_low = ~b_sig_low[yi, :, :]
        outfile = os.path.join(
            ROOT_DIR,
            f"O3_anom_year{int(y):04d}_vsLOW25_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_b_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

    print("[BlockD] All figures generated.")


if __name__ == "__main__":
    main_blockD()


In [ ]:
'''
#debug 为啥会有跳跃真是离谱，原理，25年那玩意和完整的不一样，其年末和年初不连续所以压根不能直接拼，必须计算Y-1然后拼
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Debug 脚本：检查 EXTR O3 的 daily climatology (all vs low25)
是否在 DOY=365 → DOY=1 处有奇怪跳跃。

使用数据：
  - O3_pc_EXTR_60_90N_30_70hPa_200yrs.nc
  - O3_lowest25pct_years.txt
"""

import os
import numpy as np
import xarray as xr

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"

PC_TS_EXTR_NC = os.path.join(ROOT_DIR, "O3_pc_EXTR_60_90N_30_70hPa_200yrs.nc")
LOW25_TXT     = os.path.join(ROOT_DIR, "O3_lowest25pct_years.txt")


def main():
    print("=== Debug O3 daily climatology (all vs low25) ===")
    print("PC_TS_EXTR_NC:", PC_TS_EXTR_NC)
    print("LOW25_TXT    :", LOW25_TXT)

    # 1) 读数据
    var_extr = xr.load_dataarray(PC_TS_EXTR_NC)  # (time,)
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    print("EXTR years:", years_extr[0], "→", years_extr[-1])

    n_days = int(var_extr.time.dt.dayofyear.max())
    print("n_days (should be 365):", n_days)

    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("lowest 25% O3 years:", lowest25_years)

    # 2) 计算全体 daily climatology
    clim_all = var_extr.groupby("time.dayofyear").mean("time")  # (dayofyear,)
    doys = clim_all["dayofyear"].values.astype(int)
    clim_all_vals = clim_all.values  # length 365

    # 3) 低 25% 年的 daily climatology
    var_low25 = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years))
    clim_low25 = var_low25.groupby("time.dayofyear").mean("time")
    clim_low25_vals = clim_low25.values

    # 4) 打印基本信息
    print("\nclim_all_vals shape:", clim_all_vals.shape)
    print("clim_low25_vals shape:", clim_low25_vals.shape)
    print("doys[0..10]:", doys[:10])
    print("doys[last 10]:", doys[-10:])

    # 5) 特别关注 DOY = 365 和 1 的值
    # dayofyear 是 1..365，因此：
    idx_doy1 = np.where(doys == 1)[0][0]
    idx_doy365 = np.where(doys == 365)[0][0]

    print("\n=== DOY=365 vs DOY=1 (all years) ===")
    print(f"clim_all[DOY=365] = {clim_all_vals[idx_doy365]:.4f} DU")
    print(f"clim_all[DOY=1]   = {clim_all_vals[idx_doy1]:.4f} DU")
    print(f"diff (1 - 365)    = {clim_all_vals[idx_doy1] - clim_all_vals[idx_doy365]:.4f} DU")

    print("\n=== DOY=365 vs DOY=1 (low25 years) ===")
    print(f"clim_low25[DOY=365] = {clim_low25_vals[idx_doy365]:.4f} DU")
    print(f"clim_low25[DOY=1]   = {clim_low25_vals[idx_doy1]:.4f} DU")
    print(f"diff (1 - 365)      = {clim_low25_vals[idx_doy1] - clim_low25_vals[idx_doy365]:.4f} DU")

    # 6) 如果你想直接看完整 365 个值，也可以直接 print
    #    （注意会比较长，我这里分两段打印）
    np.set_printoptions(precision=4, suppress=True, linewidth=120)

    print("\n--- clim_all_vals[0:20] (DOY 1..20) ---")
    print(clim_all_vals[0:20])

    print("\n--- clim_all_vals[-20:] (last 20 days) ---")
    print(clim_all_vals[-20:])

    print("\n--- clim_low25_vals[0:20] (DOY 1..20, low25) ---")
    print(clim_low25_vals[0:20])

    print("\n--- clim_low25_vals[-20:] (last 20 days, low25) ---")
    print(clim_low25_vals[-20:])

    print("\nDone.")

if __name__ == "__main__":
    main()
'''

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# ============================================================
# BlockE — Apply FDR to BlockC significance (Wilks 2016 / BH)
#
# Input:
#   - same EXTR O3 files & merged file as BlockC
#   - O3_lowest25pct_years.txt (from BlockA)
#   - O3_vert_anom_special.nc (from BlockC) for anomaly/clim reuse
#
# Output:
#   O3_vert_anom_special_FDR.nc
#   Contains:
#     - all vars from BlockC
#     - p-value fields for t-test & bootstrap
#     - FDR-adjusted significance masks
# ============================================================

import os, glob
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
os.makedirs(ROOT_DIR, exist_ok=True)

EXTR_O3_PATTERN = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/atm/hist/h1/o3/"
    "CO2x1SmidEmin_yBWCN.cam.h1.????.O3.isobar.nc"
)
MERGED_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)
LOW25_TXT = os.path.join(ROOT_DIR, "O3_lowest25pct_years.txt")

IN_NC  = os.path.join(ROOT_DIR, "O3_vert_anom_special.nc")  # from BlockC
OUT_NC = os.path.join(ROOT_DIR, "O3_vert_anom_special_FDR.nc")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)

ALPHA_RAW = 0.05
FDR_LEVEL = 0.10
NBOOT = 10000  # per Wilks-like practice, you cited 10000


def calc_polar_cap_o3_lev(ds, var="O3"):
    da = ds[var]
    if "lon" in da.dims:
        da = da.mean(dim="lon")
    da = da.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(da.lat))
    da = da.weighted(weights).mean(dim="lat")
    if "plev" in da.dims:
        lev_name = "plev"
    elif "lev" in da.dims:
        lev_name = "lev"
    elif "level" in da.dims:
        lev_name = "level"
    else:
        raise ValueError(f"O3 has no vertical dim, dims={da.dims}")
    if lev_name != "lev":
        da = da.rename({lev_name: "lev"})
    return da  # (time, lev)


def bootstrap_p_and_ci(col, obs, nboot=10000, alpha=0.05, rng=None):
    """
    Classic bootstrap on baseline anomalies col (1D):
      - resample with replacement
      - compute mean each boot
    Returns:
      p_boot (two-sided empirical p for obs vs boot-mean dist)
      ci_low, ci_high
    """
    col = np.asarray(col, float)
    col = col[np.isfinite(col)]
    n = col.size
    if n < 2 or not np.isfinite(obs):
        return np.nan, np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot, float)
    for i in range(nboot):
        samp = rng.choice(col, size=n, replace=True)
        means[i] = samp.mean()

    # CI of baseline mean
    lo = np.percentile(means, 100*alpha/2)
    hi = np.percentile(means, 100*(1-alpha/2))

    # empirical two-sided p for obs
    p_emp = (np.sum(np.abs(means) >= np.abs(obs)) + 1) / (nboot + 1)
    return p_emp, lo, hi


def compute_pvals_for_baseline(base_anom, anom_ref, doy_base, doy_comp,
                               nboot=10000, alpha=0.05, seed=2025):
    """
    Same loop structure as BlockC, but returns p-values:
      p_t (lev, time)
      p_b (lev, time)  empirical bootstrap p-values
    """
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    p_t = np.full((lev_n, t_len), np.nan)
    p_b = np.full((lev_n, t_len), np.nan)

    rng = np.random.default_rng(seed)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue
        day_samples = base_vals[mask_day, :]  # (n_samp, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[np.isfinite(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]

            # --- t-test p-value (single obs vs mean=0 baseline) ---
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se > 0 and np.isfinite(obs):
                tstat = obs / se
                pval = 2 * (1 - t_dist.cdf(np.abs(tstat), df=col.size-1))
                p_t[li, ti] = pval

            # --- bootstrap empirical p-value ---
            p_emp, _, _ = bootstrap_p_and_ci(col, obs, nboot=nboot, alpha=alpha, rng=rng)
            p_b[li, ti] = p_emp

    return p_t, p_b


def bh_fdr_threshold(pvals_2d, fdr=0.10):
    """
    Benjamini-Hochberg FDR threshold.
    pvals_2d: (lev, time) with NaNs allowed
    Returns:
      pFDR (scalar threshold), sig_mask_fdr (same shape)
    """
    p = np.asarray(pvals_2d).ravel()
    p = p[np.isfinite(p)]
    if p.size == 0:
        return np.nan, np.zeros_like(pvals_2d, dtype=bool)

    p_sorted = np.sort(p)
    N = p_sorted.size
    crit = (np.arange(1, N+1) / N) * fdr

    ok = p_sorted <= crit
    if not np.any(ok):
        pFDR = 0.0  # no point significant under FDR
    else:
        pFDR = p_sorted[np.where(ok)[0].max()]

    sig_fdr = (pvals_2d <= pFDR)
    sig_fdr = np.where(np.isfinite(pvals_2d), sig_fdr, False)
    return pFDR, sig_fdr


def main_blockE():
    # ---- read BlockC output for anomalies/clims (avoid recomputing them) ----
    dsC = xr.open_dataset(IN_NC)
    ref_years = dsC["ref_year"].values
    lev = dsC["lev"].values
    doy_comp = dsC["dayofyear"].values.astype(int)

    anom_all = dsC["anom_all"].values / 1e6      # back to mol/mol
    anom_low = dsC["anom_low25"].values / 1e6

    clim_all_comp = dsC["clim_all_comp"].values / 1e6
    clim_low_comp = dsC["clim_low25_comp"].values / 1e6

    # Keep original masks too
    t_sig_all_raw  = dsC["t_sig_all"].values
    b_sig_all_raw  = dsC["b_sig_all"].values
    t_sig_low_raw  = dsC["t_sig_low25"].values
    b_sig_low_raw  = dsC["b_sig_low25"].values
    time_index = dsC["time_index"].values
    dsC.close()

    # ---- baseline from EXTR (same as BlockC) ----
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)

    extr_files = sorted(glob.glob(EXTR_O3_PATTERN))
    ds_extr = xr.open_mfdataset(extr_files, combine="by_coords", parallel=False)
    o3_extr_lev = calc_polar_cap_o3_lev(ds_extr, "O3")
    ds_extr.close()

    doy_extr = o3_extr_lev.time.dt.dayofyear.values.astype(int)
    n_days = int(o3_extr_lev.time.dt.dayofyear.max())

    # climatologies
    clim_all = o3_extr_lev.groupby("time.dayofyear").mean("time")

    base_low25_cur = o3_extr_lev.sel(time=o3_extr_lev.time.dt.year.isin(lowest25_years))
    base_low25_prev = o3_extr_lev.sel(time=o3_extr_lev.time.dt.year.isin(lowest25_years-1))
    clim_low25_cur = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_prev = base_low25_prev.groupby("time.dayofyear").mean("time")
    clim_low25_full = clim_low25_cur.copy()
    oct_dec = np.arange(n_days-N_PREV+1, n_days+1, dtype=int)
    clim_low25_full.loc[dict(dayofyear=oct_dec)] = clim_low25_prev.sel(dayofyear=oct_dec).values

    # baseline anomalies
    clim_all_sel = clim_all.sel(dayofyear=o3_extr_lev.time.dt.dayofyear)
    base_anom_all = o3_extr_lev - clim_all_sel

    clim_low_sel = clim_low25_full.sel(dayofyear=o3_extr_lev.time.dt.dayofyear)
    base_anom_low = o3_extr_lev - clim_low_sel

    # ---- compute pvals & FDR for each ref_year separately ----
    n_ref = len(ref_years)
    lev_n, t_len = lev.size, doy_comp.size

    p_t_all  = np.full((n_ref, lev_n, t_len), np.nan)
    p_b_all  = np.full((n_ref, lev_n, t_len), np.nan)
    p_t_low  = np.full((n_ref, lev_n, t_len), np.nan)
    p_b_low  = np.full((n_ref, lev_n, t_len), np.nan)

    t_sig_all_fdr = np.zeros((n_ref, lev_n, t_len), bool)
    b_sig_all_fdr = np.zeros((n_ref, lev_n, t_len), bool)
    t_sig_low_fdr = np.zeros((n_ref, lev_n, t_len), bool)
    b_sig_low_fdr = np.zeros((n_ref, lev_n, t_len), bool)

    pFDR_t_all = np.full(n_ref, np.nan)
    pFDR_b_all = np.full(n_ref, np.nan)
    pFDR_t_low = np.full(n_ref, np.nan)
    pFDR_b_low = np.full(n_ref, np.nan)

    for yi, y in enumerate(ref_years):
        # (lev, time) anomalies for this ref
        ref_anom_all = anom_all[yi, :, :]
        ref_anom_low = anom_low[yi, :, :]

        print(f"[BlockE] pvals for year {int(y):04d} vs ALL baseline ...")
        pta, pba = compute_pvals_for_baseline(
            base_anom_all, ref_anom_all, doy_extr, doy_comp,
            nboot=NBOOT, alpha=ALPHA_RAW, seed=2025+int(y)
        )
        print(f"[BlockE] pvals for year {int(y):04d} vs LOW25 baseline ...")
        ptl, pbl = compute_pvals_for_baseline(
            base_anom_low, ref_anom_low, doy_extr, doy_comp,
            nboot=NBOOT, alpha=ALPHA_RAW, seed=3030+int(y)
        )

        p_t_all[yi], p_b_all[yi] = pta, pba
        p_t_low[yi], p_b_low[yi] = ptl, pbl

        # FDR thresholds + masks
        pFDR_t_all[yi], t_sig_all_fdr[yi] = bh_fdr_threshold(pta, fdr=FDR_LEVEL)
        pFDR_b_all[yi], b_sig_all_fdr[yi] = bh_fdr_threshold(pba, fdr=FDR_LEVEL)
        pFDR_t_low[yi], t_sig_low_fdr[yi] = bh_fdr_threshold(ptl, fdr=FDR_LEVEL)
        pFDR_b_low[yi], b_sig_low_fdr[yi] = bh_fdr_threshold(pbl, fdr=FDR_LEVEL)

        print(f"   pFDR_t_all={pFDR_t_all[yi]:.4f}, pFDR_b_all={pFDR_b_all[yi]:.4f}, "
              f"pFDR_t_low={pFDR_t_low[yi]:.4f}, pFDR_b_low={pFDR_b_low[yi]:.4f}")

    # ---- write new Dataset ----
    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", ref_years),
            "lev": ("lev", lev),
            "time_index": ("time_index", time_index),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            # original fields (ppm)
            "anom_all": (("ref_year","lev","time_index"), anom_all*1e6),
            "anom_low25": (("ref_year","lev","time_index"), anom_low*1e6),
            "clim_all_comp": (("lev","time_index"), clim_all_comp*1e6),
            "clim_low25_comp": (("lev","time_index"), clim_low_comp*1e6),

            # raw masks from BlockC
            "t_sig_all_raw": (("ref_year","lev","time_index"), t_sig_all_raw),
            "b_sig_all_raw": (("ref_year","lev","time_index"), b_sig_all_raw),
            "t_sig_low25_raw": (("ref_year","lev","time_index"), t_sig_low_raw),
            "b_sig_low25_raw": (("ref_year","lev","time_index"), b_sig_low_raw),

            # p-value fields
            "p_t_all": (("ref_year","lev","time_index"), p_t_all),
            "p_b_all": (("ref_year","lev","time_index"), p_b_all),
            "p_t_low25": (("ref_year","lev","time_index"), p_t_low),
            "p_b_low25": (("ref_year","lev","time_index"), p_b_low),

            # FDR thresholds (per ref_year)
            "pFDR_t_all": (("ref_year",), pFDR_t_all),
            "pFDR_b_all": (("ref_year",), pFDR_b_all),
            "pFDR_t_low25": (("ref_year",), pFDR_t_low),
            "pFDR_b_low25": (("ref_year",), pFDR_b_low),

            # FDR masks
            "t_sig_all_fdr": (("ref_year","lev","time_index"), t_sig_all_fdr),
            "b_sig_all_fdr": (("ref_year","lev","time_index"), b_sig_all_fdr),
            "t_sig_low25_fdr": (("ref_year","lev","time_index"), t_sig_low_fdr),
            "b_sig_low25_fdr": (("ref_year","lev","time_index"), b_sig_low_fdr),
        },
        attrs={
            "description": (
                "BlockE output: BlockC anomalies + p-values + FDR-adjusted significance "
                "(Benjamini-Hochberg following Wilks 2016). "
                f"FDR control level = {FDR_LEVEL:.2f}, raw alpha = {ALPHA_RAW:.2f}, "
                f"bootstrap iterations = {NBOOT}."
            )
        }
    )

    ds_out["anom_all"].attrs["units"]="ppm"
    ds_out["anom_low25"].attrs["units"]="ppm"
    ds_out["clim_all_comp"].attrs["units"]="ppm"
    ds_out["clim_low25_comp"].attrs["units"]="ppm"

    ds_out.to_netcdf(OUT_NC)
    print("[BlockE] Saved:", OUT_NC)


if __name__ == "__main__":
    main_blockE()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# ============================================================
# BlockF — Plot time–height anomalies with FDR masks
#
# Same as BlockD but:
#   - read O3_vert_anom_special_FDR.nc
#   - use *_fdr masks
#   - output filenames contain "withFDR"
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT_DIR = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
os.makedirs(ROOT_DIR, exist_ok=True)

IN_NC = os.path.join(ROOT_DIR, "O3_vert_anom_special_FDR.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
                "May", "June", "July", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rc("font", family="sans-serif")
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
mpl.rc("axes", titlesize=11, labelsize=9, linewidth=0.8)
mpl.rc("legend", fontsize=7, frameon=False)
mpl.rc("figure", figsize=(6, 4), dpi=300)
mpl.rc("savefig", bbox="tight", pad_inches=0.1)


def guess_p_2_pa(lev_vals):
    lev_vals = np.asarray(lev_vals)
    if np.nanmax(lev_vals) <= 2000:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"


def plot_time_height_anom(
    x_idx, lev_vals, anom_vals, clim_vals, nonsig_mask,
    ref_year, baseline_label, test_label, outfile,
):
    fig, ax = plt.subplots()
    x = np.asarray(x_idx)
    p_pa, p_unit = guess_p_2_pa(lev_vals)

    vlim = np.nanmax(np.abs(anom_vals))
    vmax = min(vlim, 2.0) if np.isfinite(vlim) and vlim > 0 else 1.5
    levels = np.linspace(-vmax, vmax, 31)

    cf = ax.contourf(
        x, p_pa, anom_vals,
        levels=levels, cmap="RdBu_r",
        extend="both", antialiased=True, alpha=0.85,
    )

    clim_mean = np.nanmean(clim_vals)
    clim_std  = np.nanstd(clim_vals)
    if np.isfinite(clim_mean) and np.isfinite(clim_std) and clim_std > 0:
        clim_levels = np.linspace(clim_mean-1.5*clim_std, clim_mean+1.5*clim_std, 7)
    else:
        clim_levels = 7

    CS = ax.contour(x, p_pa, clim_vals, levels=clim_levels, colors="k", linewidths=0.7)
    ax.clabel(CS, inline=True, fontsize=6, fmt="%.2f")

    if nonsig_mask is not None:
        mask_int = nonsig_mask.astype(int)
        ax.contourf(
            x, p_pa, mask_int,
            levels=[0.5, 1.5],
            colors="none", hatches=["///"], alpha=0,
        )
        patch = Patch(
            facecolor="white", hatch="///", edgecolor="black",
            label="Not significant after FDR",
        )
        ax.legend(handles=[patch], loc="upper right")

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel(f"Pressure ({p_unit})")

    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)

    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.5)

    ax.set_title(
        f"O$_3$ anomaly (ppm), Year {int(ref_year):04d}\n"
        f"Baseline: {baseline_label}, Mask: non-significant by {test_label} + FDR"
    )

    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("O$_3$ anomaly (ppm)")

    plt.savefig(outfile, dpi=300)
    plt.close(fig)
    print("  Saved:", outfile)


def main_blockF():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_index = ds["time_index"].values

    anom_all = ds["anom_all"].values
    anom_low = ds["anom_low25"].values
    clim_all = ds["clim_all_comp"].values
    clim_low = ds["clim_low25_comp"].values

    t_sig_all = ds["t_sig_all_fdr"].values
    b_sig_all = ds["b_sig_all_fdr"].values
    t_sig_low = ds["t_sig_low25_fdr"].values
    b_sig_low = ds["b_sig_low25_fdr"].values

    ds.close()

    for yi, y in enumerate(ref_years):
        nonsig_t_all = ~t_sig_all[yi]
        outfile = os.path.join(ROOT_DIR, f"O3_anom_year{int(y):04d}_vsALL_t_withFDR.png")
        plot_time_height_anom(
            time_index, lev, anom_all[yi], clim_all, nonsig_t_all,
            ref_year=y, baseline_label="EXTR all-years climatology",
            test_label="t-test", outfile=outfile,
        )

        nonsig_b_all = ~b_sig_all[yi]
        outfile = os.path.join(ROOT_DIR, f"O3_anom_year{int(y):04d}_vsALL_boot_withFDR.png")
        plot_time_height_anom(
            time_index, lev, anom_all[yi], clim_all, nonsig_b_all,
            ref_year=y, baseline_label="EXTR all-years climatology",
            test_label="bootstrap", outfile=outfile,
        )

        nonsig_t_low = ~t_sig_low[yi]
        outfile = os.path.join(ROOT_DIR, f"O3_anom_year{int(y):04d}_vsLOW25_t_withFDR.png")
        plot_time_height_anom(
            time_index, lev, anom_low[yi], clim_low, nonsig_t_low,
            ref_year=y, baseline_label="EXTR low-25% climatology",
            test_label="t-test", outfile=outfile,
        )

        nonsig_b_low = ~b_sig_low[yi]
        outfile = os.path.join(ROOT_DIR, f"O3_anom_year{int(y):04d}_vsLOW25_boot_withFDR.png")
        plot_time_height_anom(
            time_index, lev, anom_low[yi], clim_low, nonsig_b_low,
            ref_year=y, baseline_label="EXTR low-25% climatology",
            test_label="bootstrap", outfile=outfile,
        )

    print("[BlockF] All FDR figures generated.")


if __name__ == "__main__":
    main_blockF()
